In [1]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

In [ ]:
base_dir = Path("codon_opt_benchmark-1/codon_opt_results_m13").resolve()
mutate_probs = [
    0.0005, 0.0006, 0.0007, 0.0008, 0.0009,
    0.0010, 0.0011, 0.0012, 0.0013, 0.0014,
    0.0015, 0.0016, 0.0017, 0.0018, 0.0019,
    0.0020, 0.0021, 0.0022, 0.0023, 0.0024,
]
target_name = "M13_best_per_iteration.csv"
records = []

for prob in mutate_probs:
    folder = base_dir / f"mut_{prob:.4f}".rstrip("0").rstrip(".")
    csv_path = folder / target_name
    if not csv_path.exists():
        print(f"Missing file: {csv_path}")
        continue
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f"Failed to read {csv_path}: {exc}")
        continue
    if {"iteration", "score"}.issubset(df.columns):
        df = df[["iteration", "score"]].copy()
        df["mutate_prob"] = prob
        records.append(df)
    else:
        print(f"Unexpected columns in {csv_path}: {df.columns.tolist()}")

if records:
    merged_df = (
        pd.concat(records, ignore_index=True)
        .sort_values(["mutate_prob", "iteration"])
        .reset_index(drop=True)
    )
else:
    merged_df = pd.DataFrame(columns=["iteration", "score", "mutate_prob"])

print(f"Loaded {len(records)} mutation bins, {len(merged_df):,} rows total.")
merged_df.head()

Loaded 0 mutation bins, 0 rows total.


,iteration,score,mutate_prob


In [3]:
if merged_df.empty:
    print("No data found. Check that mutation-run folders and best-per-iteration CSVs exist.")
else:
    plt.figure(figsize=(10, 6))
    ax = sns.lineplot(
        data=merged_df, x="iteration", y="score", hue="mutate_prob", palette="turbo", linewidth=1
    )
    ax.set_title("Best-per-iteration scores across mutation probabilities")
    ax.set_ylabel("Score")
    ax.set_xlabel("Iteration")
    ax.legend(title="mutate_prob", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

No data found. Check that mutation-run folders and best-per-iteration CSVs exist.
